# Phase 3A: Feature Engineering & Data Preparation

**Project**: Neural PK-PD Modeling with Physics-Informed Neural ODEs  
**Date**: February 24, 2026  
**Previous Phase**: [phase1_2_data_exploration.ipynb](phase1_2_data_exploration.ipynb)  
**Next Phase**: [phase3b_model_design.ipynb](phase3b_model_design.ipynb)  

---

## 🎯 Phase 3A Objectives

1. **Environment Setup** — Import libraries and configure reproducibility
2. **Configuration** — Define hyperparameters and model settings
3. **Data Loading** — Load Phase 2 processed multi-task features
4. **Feature Engineering** — Extract per-task feature matrices and build DataLoaders
5. **Data Quality Gate** — Validate splits for leakage, NaN/Inf, class balance
6. **Save Artifacts** — Persist processed data for the model design notebook


---
## 1. Environment Setup & Imports

In [ ]:
# =============================================================================
# CELL: Execution Timestamp Logger (with start + end tracking)
# PURPOSE: Record session start, environment info, and capture cell-level
#          execution timestamps (start + end + duration) for a full timeline
# INPUTS:  None
# OUTPUTS: Prints session header; provides log_cell_start() / log_cell_end()
# =============================================================================
import datetime
import platform
import sys

# -- Execution log -----------------------------------------------------------
CELL_EXEC_LOG: list = []   # [{section, start, end, duration_s}, ...]


def log_cell_start(section: str) -> None:
    """Record cell execution start time."""
    now = datetime.datetime.now()
    CELL_EXEC_LOG.append({"section": section, "start": now, "end": None, "duration_s": None})
    print(f"\u23f1  [{now.strftime('%Y-%m-%d %H:%M:%S')}] Starting: {section}")


def log_cell_end() -> None:
    """Record cell execution end time for the most recent entry."""
    now = datetime.datetime.now()
    if CELL_EXEC_LOG and CELL_EXEC_LOG[-1]["end"] is None:
        entry = CELL_EXEC_LOG[-1]
        entry["end"] = now
        entry["duration_s"] = round((now - entry["start"]).total_seconds(), 2)
        print(f"\u23f1  [{now.strftime('%Y-%m-%d %H:%M:%S')}] Completed in {entry['duration_s']:.1f}s")


def log_elapsed(label: str) -> None:
    """Print wall-clock elapsed time since SESSION_START."""
    elapsed = (datetime.datetime.now() - SESSION_START).total_seconds()
    h, rem = divmod(int(elapsed), 3600)
    m, s = divmod(rem, 60)
    print(f"\u23f1  [{label}] Elapsed: {h}h {m}m {s}s")


# -- Session start time ------------------------------------------------------
SESSION_START = datetime.datetime.now()

# -- Session header -----------------------------------------------------------
print("=" * 65)
print("  PHASE 3A -- Feature Engineering & Data Preparation")
print("  Execution Timestamp Logger")
print("=" * 65)
print(f"  Started     : {SESSION_START.strftime('%Y-%m-%d  %H:%M:%S')}")
print(f"  Python      : {sys.version.split()[0]}  ({platform.python_implementation()})")
print(f"  Platform    : {platform.system()} {platform.release()} ({platform.machine()})")

import torch
print(f"  PyTorch     : {torch.__version__}")
device_name = (
    torch.cuda.get_device_name(0) if torch.cuda.is_available()
    else ("MPS" if torch.backends.mps.is_available() else "CPU (no GPU)")
)
print(f"  Device      : {device_name}")

try:
    import rdkit
    print(f"  RDKit       : {rdkit.__version__}")
except ImportError:
    print("  RDKit       : NOT FOUND")

try:
    import torchdiffeq
    print(f"  torchdiffeq : {torchdiffeq.__version__}")
except (ImportError, AttributeError):
    print("  torchdiffeq : installed (version attr unavailable)")

print("=" * 65)
print("  Notebook: phase3a_feature_engineering.ipynb")
print("  Project : Neural PK-PD Modeling with ODE")
print("=" * 65)
print("\u2705 Timestamp logger ready  ->  log_cell_start() / log_cell_end() active")
print("   Run the timeline summary cell at the end to view the full log.")

In [ ]:
# Core Libraries

log_cell_start("Core Libraries & Imports")

import numpy as np
import pandas as pd
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# Deep Learning
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset

# Neural ODE
from torchdiffeq import odeint

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error, r2_score, mean_absolute_error,
    roc_auc_score, accuracy_score, f1_score, classification_report, roc_curve
)

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

log_cell_end()


---
## 2. Configuration & Hyperparameters

In [ ]:

# ============================================
# HYPERPARAMETERS
# ============================================
log_cell_start("Cell 02 - Configuration")

N_BITS = 1024  # default; overwritten by processed feature columns
PHYSICO_DIM = 2

config = {
    # Data
    'test_size': 0.2,
    'val_size': 0.1,

    # Model Architecture
    'input_dim': PHYSICO_DIM + N_BITS,
    'hidden_dim': 128,
    'latent_dim': 64,
    'dropout': 0.2,
    'reg_head_hidden': 64,    # deeper head hidden width for hard regression tasks
    'reg_head_dropout': 0.1,

    # Training
    'batch_size': 64,
    'epochs': 300,
    'learning_rate': 1e-3,
    'weight_decay': 1e-4,
    'patience': 40,
    'grad_clip': 1.0,

    # Multi-task Loss Weights
    'w_binding': 1.8,
    'w_herg': 1.0,
    'w_caco2': 2.0,
    'w_clearance': 1.0,
    'w_physics': 0.1,

    # Classification loss settings
    'classification_use_logits': True,
    'use_focal_for_classification': True,
    'focal_gamma': 2.0,
    'herg_pos_weight': 2.5,   # overwritten from data in Cell 8
    'caco2_pos_weight': 1.0,  # overwritten from data in Cell 8
}

print("Configuration loaded:")
for k, v in config.items():
    print(f"  {k}: {v}")

log_cell_end()


---
## 3. Data Loading

Load preprocessed data from Phase 1-2. See [phase1_2_data_exploration.ipynb](phase1_2_data_exploration.ipynb) for data processing details.

In [ ]:

# ============================================
# LOAD PHASE 2 PROCESSED FEATURES
# ============================================
log_cell_start("Cell 03 - Dataset Loading")

CODING_ROOT = next(
    (candidate for candidate in [Path.cwd(), *Path.cwd().parents] if (candidate / 'data').exists()),
    Path.cwd(),
)
DATA_DIR = CODING_ROOT / 'data/raw'
PHASE2_FEATURE_PATH = CODING_ROOT / 'data/processed/phase2_multitask_features_with_fingerprints.csv'

phase2_features = pd.read_csv(PHASE2_FEATURE_PATH)
print(f"Loaded Phase 2 features: {len(phase2_features)} rows")
print(f"Columns: {len(phase2_features.columns)}")
print(f"Tasks: {phase2_features['task'].value_counts().to_dict()}")

log_cell_end()


In [ ]:

# ============================================
# INSPECT PROCESSED FEATURE STRUCTURE
# ============================================
log_cell_start("Cell 04 - Dataset Inspection")

fingerprint_cols_preview = [col for col in phase2_features.columns if col.startswith('fp_')]
print("Processed feature columns (first 20):")
print(phase2_features.columns.tolist()[:20])
print(f"\nFingerprint column count: {len(fingerprint_cols_preview)}")
print(f"Task labels: {sorted(phase2_features['task'].unique().tolist())}")

log_cell_end()


---
## 4. Feature Engineering & Preprocessing

In [ ]:

# ============================================
# PREPARE FEATURES FOR EACH TASK
# ============================================
log_cell_start("Cell 05 - Task Data Prep Fn")

def prepare_task_data(df, feature_cols, target_col, task_type='regression'):
    """
    Prepare features and targets for a specific task.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Input dataframe
    feature_cols : list
        Column names for features
    target_col : str
        Column name for target
    task_type : str
        'regression' or 'classification'
        
    Returns:
    --------
    X, y : numpy arrays
    """
    # Drop rows with missing values
    subset_cols = feature_cols + [target_col]
    df_clean = df.dropna(subset=subset_cols)
    
    X = df_clean[feature_cols].values
    y = df_clean[target_col].values
    
    print(f"  Samples: {len(X)}, Features: {X.shape[1]}, Target: {target_col}")
    
    return X, y

print("Feature preparation function defined.")

log_cell_end()


In [ ]:

# ============================================
# FEATURE SPACE SETUP FROM PHASE 2 OUTPUT
# ============================================
log_cell_start("Cell 06 - Feature Space Setup")

PHYSICO_FEATURES = ['MW', 'LogP']
FINGERPRINT_COLS = sorted([col for col in phase2_features.columns if col.startswith('fp_')])
if len(FINGERPRINT_COLS) == 0:
    raise ValueError("No fingerprint columns found in Phase 2 processed file.")

N_BITS = len(FINGERPRINT_COLS)
FEATURE_COLS = PHYSICO_FEATURES + FINGERPRINT_COLS
FEAT_DIM = len(FEATURE_COLS)
config['input_dim'] = FEAT_DIM

print("Using processed features from Phase 2:")
print(f"  Physico features: {len(PHYSICO_FEATURES)} -> {PHYSICO_FEATURES}")
print(f"  Fingerprint bits: {N_BITS}")
print(f"  Total input dim:  {FEAT_DIM}")

log_cell_end()


In [ ]:

# ============================================
# EXTRACT FEATURES PER TASK (FROM PHASE 2 MATRIX)
# ============================================
log_cell_start("Cell 07 - Feature Extraction")

def extract_task_matrix(task_name):
    task_df = phase2_features[phase2_features['task'] == task_name].copy()
    if task_df.empty:
        raise ValueError(f"No rows found for task: {task_name}")
    X = task_df[FEATURE_COLS].values.astype(np.float32)
    y = task_df['target'].values.astype(np.float32)
    return X, y

X_binding, y_binding = extract_task_matrix('binding_affinity')
print(f"Task 1 Binding   — samples: {len(X_binding)}, features: {X_binding.shape[1]}")

X_herg, y_herg = extract_task_matrix('hERG_inhibition')
print(f"Task 2 hERG      — samples: {len(X_herg)}, features: {X_herg.shape[1]}, pos%: {y_herg.mean():.1%}")

X_caco2, y_caco2 = extract_task_matrix('Caco2_permeability')
y_caco2 = (y_caco2 > 0.5).astype(np.float32)
print(f"Task 3 Caco-2    — samples: {len(X_caco2)}, features: {X_caco2.shape[1]}, pos%: {y_caco2.mean():.1%}")

X_clearance, y_clearance = extract_task_matrix('hepatocyte_clearance')
print(f"Task 4 Clearance — samples: {len(X_clearance)}, features: {X_clearance.shape[1]}")

print("\nAll tasks prepared from Phase 2 processed matrix.")

log_cell_end()


In [ ]:

# ============================================
# BUILD TRAIN / VAL / TEST DATA LOADERS
# ============================================
log_cell_start("Cell 08 - DataLoaders")

from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def make_loaders(X, y, config, is_regression=True, random_state=42):
    """
    Split into train/val/test and return DataLoaders with {'X', 'y'} batches.
    Normalizes both features AND targets (for regression) on the train split only.
    """
    X_tv, X_test, y_tv, y_test = train_test_split(
        X, y, test_size=config['test_size'], random_state=random_state
    )
    val_frac = config['val_size'] / (1 - config['test_size'])
    X_train, X_val, y_train, y_val = train_test_split(
        X_tv, y_tv, test_size=val_frac, random_state=random_state
    )

    # Normalize features (fit on train only)
    feat_scaler = StandardScaler()
    X_train = feat_scaler.fit_transform(X_train)
    X_val   = feat_scaler.transform(X_val)
    X_test  = feat_scaler.transform(X_test)

    # Normalize regression targets (classification targets stay as 0/1)
    target_scaler = None
    if is_regression:
        target_scaler = StandardScaler()
        y_train = target_scaler.fit_transform(y_train.reshape(-1, 1)).flatten()
        y_val   = target_scaler.transform(y_val.reshape(-1, 1)).flatten()
        y_test  = target_scaler.transform(y_test.reshape(-1, 1)).flatten()

    def to_loader(Xa, ya, shuffle=True):
        ds = TensorDataset(
            torch.FloatTensor(Xa),
            torch.FloatTensor(ya).unsqueeze(1)
        )
        def collate(batch):
            Xb, yb = zip(*batch)
            return {'X': torch.stack(Xb), 'y': torch.stack(yb)}
        return DataLoader(ds, batch_size=config['batch_size'],
                          shuffle=shuffle, collate_fn=collate)

    return (
        to_loader(X_train, y_train, shuffle=True),
        to_loader(X_val,   y_val,   shuffle=False),
        to_loader(X_test,  y_test,  shuffle=False),
        feat_scaler,
        target_scaler,
    )

def compute_pos_weight(y_binary):
    y_binary = np.asarray(y_binary).astype(np.float32)
    pos = float((y_binary == 1).sum())
    neg = float((y_binary == 0).sum())
    if pos == 0:
        return 1.0
    return max(1.0, neg / pos)

# Regression / classification flags per task
task_types = {
    'binding':   True,   # regression
    'herg':      False,  # classification
    'caco2':     False,  # classification
    'clearance': True,   # regression
}

tasks_raw = {
    'binding':   (X_binding,   y_binding),
    'herg':      (X_herg,      y_herg),
    'caco2':     (X_caco2,     y_caco2),
    'clearance': (X_clearance, y_clearance),
}

# Auto-tune class weights from current task labels
config['herg_pos_weight'] = compute_pos_weight(y_herg)
config['caco2_pos_weight'] = compute_pos_weight(y_caco2)

train_loaders, val_loaders, test_loaders = {}, {}, {}
feat_scalers, target_scalers = {}, {}

for task, (X, y) in tasks_raw.items():
    tr, va, te, fs, ts = make_loaders(X, y, config, is_regression=task_types[task])
    train_loaders[task]   = tr
    val_loaders[task]     = va
    test_loaders[task]    = te
    feat_scalers[task]    = fs
    target_scalers[task]  = ts   # None for classification

print("DataLoaders created (regression targets normalized to mean=0, std=1):")
for task, loader in train_loaders.items():
    n = len(loader.dataset)
    reg = "regression" if task_types[task] else "classification"
    print(f"  {task:<12}  {reg:<14}  train={n}, "
          f"val={len(val_loaders[task].dataset)}, "
          f"test={len(test_loaders[task].dataset)}")

print("\nAuto class weights:")
print(f"  herg_pos_weight:  {config['herg_pos_weight']:.3f}")
print(f"  caco2_pos_weight: {config['caco2_pos_weight']:.3f}")

log_cell_end()


---
## 4.5 Data Quality Gate (Run Before Training)

Use this concrete check cell to validate:
- missing/invalid values in features and targets
- duplicate feature rows within each task
- split leakage (exact feature overlap across train/val/test)
- class balance (classification) and target-distribution stability (regression)

If any leakage is non-zero, fix split strategy before interpreting model metrics.

In [ ]:

# ============================================
# DATA QUALITY CHECKS (concrete gate)
# ============================================
log_cell_start("Cell 08A - Data Quality Gate")

from sklearn.model_selection import train_test_split


def _split_like_training(X, y, cfg, random_state=42):
    X_tv, X_test, y_tv, y_test = train_test_split(
        X, y, test_size=cfg['test_size'], random_state=random_state
    )
    val_frac = cfg['val_size'] / (1 - cfg['test_size'])
    X_train, X_val, y_train, y_val = train_test_split(
        X_tv, y_tv, test_size=val_frac, random_state=random_state
    )
    return X_train, X_val, X_test, y_train, y_val, y_test


def _row_hash(X):
    Xc = np.ascontiguousarray(X)
    return Xc.view(np.dtype((np.void, Xc.dtype.itemsize * Xc.shape[1]))).ravel()


quality_rows = []

for task, (X_task, y_task) in tasks_raw.items():
    X_task = np.asarray(X_task)
    y_task = np.asarray(y_task)

    n_samples, n_features = X_task.shape
    n_nan_X = int(np.isnan(X_task).sum())
    n_inf_X = int(np.isinf(X_task).sum())
    n_nan_y = int(np.isnan(y_task).sum())
    n_inf_y = int(np.isinf(y_task).sum())

    unique_rows = np.unique(X_task, axis=0).shape[0]
    dup_rows = int(n_samples - unique_rows)
    dup_pct = 100.0 * dup_rows / n_samples

    X_tr, X_va, X_te, y_tr, y_va, y_te = _split_like_training(X_task, y_task, config, random_state=42)

    h_tr = np.unique(_row_hash(X_tr))
    h_va = np.unique(_row_hash(X_va))
    h_te = np.unique(_row_hash(X_te))

    leak_tr_va = int(np.intersect1d(h_tr, h_va).size)
    leak_tr_te = int(np.intersect1d(h_tr, h_te).size)
    leak_va_te = int(np.intersect1d(h_va, h_te).size)

    if task_types[task]:
        # Regression checks
        tr_mu, va_mu, te_mu = float(np.mean(y_tr)), float(np.mean(y_va)), float(np.mean(y_te))
        tr_sd = float(np.std(y_tr) + 1e-8)
        drift_va = abs(va_mu - tr_mu) / tr_sd
        drift_te = abs(te_mu - tr_mu) / tr_sd

        quality_rows.append({
            'task': task,
            'type': 'regression',
            'samples': n_samples,
            'features': n_features,
            'nan_inf_total': n_nan_X + n_inf_X + n_nan_y + n_inf_y,
            'dup_rows': dup_rows,
            'dup_pct': round(dup_pct, 2),
            'leak_tr_val': leak_tr_va,
            'leak_tr_test': leak_tr_te,
            'leak_val_test': leak_va_te,
            'train_mean': round(tr_mu, 4),
            'val_mean': round(va_mu, 4),
            'test_mean': round(te_mu, 4),
            'val_drift_sigma': round(drift_va, 3),
            'test_drift_sigma': round(drift_te, 3),
        })
    else:
        # Classification checks
        tr_pos = float(np.mean(y_tr))
        va_pos = float(np.mean(y_va))
        te_pos = float(np.mean(y_te))

        quality_rows.append({
            'task': task,
            'type': 'classification',
            'samples': n_samples,
            'features': n_features,
            'nan_inf_total': n_nan_X + n_inf_X + n_nan_y + n_inf_y,
            'dup_rows': dup_rows,
            'dup_pct': round(dup_pct, 2),
            'leak_tr_val': leak_tr_va,
            'leak_tr_test': leak_tr_te,
            'leak_val_test': leak_va_te,
            'train_pos_rate': round(tr_pos, 4),
            'val_pos_rate': round(va_pos, 4),
            'test_pos_rate': round(te_pos, 4),
            'val_pos_delta': round(abs(va_pos - tr_pos), 4),
            'test_pos_delta': round(abs(te_pos - tr_pos), 4),
        })

quality_df = pd.DataFrame(quality_rows)

print("\n✅ Data quality summary by task")
display(quality_df)

# Concrete gate rules
nan_inf_ok = bool((quality_df['nan_inf_total'] == 0).all())
leakage_ok = bool(((quality_df['leak_tr_val'] == 0) & (quality_df['leak_tr_test'] == 0) & (quality_df['leak_val_test'] == 0)).all())

if 'val_drift_sigma' in quality_df.columns:
    drift_cols = [c for c in ['val_drift_sigma', 'test_drift_sigma'] if c in quality_df.columns]
    drift_ok = bool((quality_df[drift_cols].fillna(0) <= 0.5).all().all())
else:
    drift_ok = True

if 'val_pos_delta' in quality_df.columns:
    delta_cols = [c for c in ['val_pos_delta', 'test_pos_delta'] if c in quality_df.columns]
    class_balance_ok = bool((quality_df[delta_cols].fillna(0) <= 0.10).all().all())
else:
    class_balance_ok = True

print("\n📌 QUALITY GATE RESULTS")
print(f"  No NaN/Inf values:                {'PASS' if nan_inf_ok else 'FAIL'}")
print(f"  No split leakage (exact overlap): {'PASS' if leakage_ok else 'FAIL'}")
print(f"  Regression drift <= 0.5σ:         {'PASS' if drift_ok else 'FAIL'}")
print(f"  Class balance delta <= 0.10:      {'PASS' if class_balance_ok else 'FAIL'}")

all_pass = nan_inf_ok and leakage_ok and drift_ok and class_balance_ok
print(f"\n🎯 DATA QUALITY GATE: {'PASS' if all_pass else 'FAIL'}")
if not all_pass:
    print("   Action: fix leakage/split/imbalance issues before trusting benchmark gains.")

log_cell_end()


---
## 5. Save Artifacts for Model Design Notebook

Save all processed data, scalers, configuration, and feature metadata so that
`phase3b_model_design.ipynb` can load them directly without re-running feature engineering.


In [ ]:
# ============================================
# SAVE PROCESSED ARTIFACTS FOR PHASE 3B
# ============================================
log_cell_start("Cell 09 - Save Artifacts")

import pickle, json as _json

ARTIFACT_DIR = CODING_ROOT / 'data/processed/phase3a_artifacts'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# 1. Save task arrays (features + targets)
with open(ARTIFACT_DIR / 'tasks_raw.pkl', 'wb') as f:
    pickle.dump(tasks_raw, f)

# 2. Save task type flags
with open(ARTIFACT_DIR / 'task_types.pkl', 'wb') as f:
    pickle.dump(task_types, f)

# 3. Save config dict
with open(ARTIFACT_DIR / 'config.json', 'w') as f:
    _json.dump(config, f, indent=2)

# 4. Save DataLoaders components (scalers)
loader_data = {
    'feat_scalers': feat_scalers,
    'target_scalers': target_scalers,
)}
with open(ARTIFACT_DIR / 'scalers.pkl', 'wb') as f:
    pickle.dump(loader_data, f)

# 5. Save feature column names
feature_meta = {
    'FEATURE_COLS': FEATURE_COLS,
    'PHYSICO_FEATURES': PHYSICO_FEATURES,
    'FINGERPRINT_COLS': FINGERPRINT_COLS,
    'N_BITS': N_BITS,
    'FEAT_DIM': FEAT_DIM,
)}
with open(ARTIFACT_DIR / 'feature_meta.pkl', 'wb') as f:
    pickle.dump(feature_meta, f)

# 6. Save quality gate results
quality_df.to_csv(ARTIFACT_DIR / 'quality_gate.csv', index=False)

# 7. Save the Phase 2 features DataFrame for downstream sections
phase2_features.to_parquet(ARTIFACT_DIR / 'phase2_features.parquet', index=False)

print(f'\n✅ Phase 3A artifacts saved to {ARTIFACT_DIR}/')
for artifact_path in sorted(ARTIFACT_DIR.iterdir()):
    fsize = artifact_path.stat().st_size
    print(f"  {artifact_path.name:<35} {fsize / 1024:>8.1f} KB")

log_cell_end()

In [ ]:

# ============================================
# EXECUTION TIMELINE SUMMARY
# ============================================
print("=" * 75)
print("  PHASE 3A — EXECUTION TIMELINE SUMMARY")
print("=" * 75)
print(f"  {'#':<4} {'Section':<35} {'Start':>10} {'End':>10} {'Duration':>10}")
print("-" * 75)

for i, entry in enumerate(CELL_EXEC_LOG, 1):
    section = entry["section"][:34]
    start   = entry["start"].strftime("%H:%M:%S") if entry["start"] else "—"
    end     = entry["end"].strftime("%H:%M:%S") if entry["end"] else "running…"
    dur     = f"{entry['duration_s']:.1f}s" if entry["duration_s"] is not None else "—"
    print(f"  {i:<4} {section:<35} {start:>10} {end:>10} {dur:>10}")

total_s = sum(e["duration_s"] for e in CELL_EXEC_LOG if e["duration_s"] is not None)
wall_s  = (datetime.datetime.now() - SESSION_START).total_seconds()

print("-" * 75)
print(f"  {'Total cell time:':<40} {total_s:>28.1f}s")
print(f"  {'Wall-clock time:':<40} {wall_s:>28.1f}s")
print(f"  {'Session started:':<40} {SESSION_START.strftime('%Y-%m-%d %H:%M:%S'):>28}")
print(f"  {'Summary generated:':<40} {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'):>28}")
print("=" * 75)
